# Physical AI 第8回実習 v06：新しい日本語72指示による独立評価

研究段階：**独立評価（新規の合成テキスト72件・暫定ラベル）**。v02～v05の開発用72件は**モデル設定の比較・閾値選択に使用済み**なので、今回の性能値の独立評価には使用しない。

- 第7回Sentence Transformerのタスク推論・語彙ゲート、およびv04の優先拒否ゲートを再現する。
- 比較する2条件は**評価前に固定**する：A：score 0.60 / margin 0.025、B：score 0.35 / margin 0.010（v05開発データによる候補）。
- 第8回v06の72件は旧データと文章一致がないことを事前検証。**今回のモデル出力はまだ取得していない。**
- 「独立」とは旧72件の予測結果によるスコア再利用をしないこと。AIが作成した合成文章であり、現場運用への一般化、統計的無作為抽出、安全性を意味しない。
- `expected_action` は運用定義に従った**事前提案ラベル**で、人間の正式確定前。ラベル修正が必要なら**推論実行前**に別版を作る。
- このNotebookは機能追加やロボット制御を行わない。01～07のNotebookは一切変更しない。

## セル1：v06データの読み込み・事前登録の確認

`PhysicalAI_08_v06_independent_evaluation.zip` をWindowsに保存してください。ノートブックをColabで開いた後、このセルを実行し、ファイル選択が出たら **ZIP** をアップロードします。v05 ZIPは選択しません。

以下のコードは、CSVの件数・旧72件との文字列重複・SHA256を検査し、ラベル分布と人間が確認するための例文を表示します。ここではまだAIを実行しません。

In [ ]:
from pathlib import Path
import csv, hashlib, zipfile, os, sys, re, unicodedata, json, math, platform
from datetime import datetime, timezone
from collections import Counter
ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
VERSION='v06'
HOLDOUT='08_v06_holdout_72_preregistered.csv'
OLD='08_text_testset_v02_READONLY.csv'
PRED02='08_text_predictions_v02_READONLY.csv'
PRED04='08_text_predictions_replayed_v04_READONLY.csv'
EXPECTED_HOLDOUT_SHA='d8ab45ff83325c4c26a457eef26d29f07735d349fe1a903752dc908144bd029c'
EXPECTED_V02_SHA='56de4f7a1f06bf3963376998328c9547dd2e45e0a1107cf64544f787ebe8b0c5'
EXPECTED_V04_SHA='3d730903bde15aa97aaa824cb68a9856f60a9889da5aaed087e03f58b4965279'

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def load_csv(name):
    with (ROOT/name).open('r',newline='',encoding='utf-8-sig') as fh:
        return list(csv.DictReader(fh))

if not (ROOT/HOLDOUT).exists():
    try:
        from google.colab import files
        print('v06 ZIPをアップロードしてください。v05 ZIPではありません。')
        incoming=files.upload()
        candidates=[n for n in incoming if n.lower().endswith('.zip')]
        if len(candidates)!=1: raise RuntimeError('v06のZIPファイルを1個だけ選択してください')
        with zipfile.ZipFile(ROOT/candidates[0]) as archive:
            allowed={HOLDOUT,OLD,PRED02,PRED04,'08_v06_old72_label_audit.csv'}
            for item in archive.infolist():
                if item.filename in allowed:
                    (ROOT/item.filename).write_bytes(archive.read(item))
    except ImportError as e:
        raise RuntimeError('Colab以外では、ZIPのCSVをNotebookと同じ場所へ配置してください') from e
for f in (HOLDOUT,OLD,PRED02,PRED04):
    if not (ROOT/f).exists(): raise FileNotFoundError(f'必要ファイルがありません: {f}。v06 ZIPを指定してください')

holdout=load_csv(HOLDOUT)
old=load_csv(OLD)
ref02=load_csv(PRED02)
ref04=load_csv(PRED04)
assert len(holdout)==len(old)==len(ref02)==len(ref04)==72
assert sha256(ROOT/HOLDOUT)==EXPECTED_HOLDOUT_SHA, '評価文が変更されています。修正版は別バージョンにしてください。'
assert sha256(ROOT/PRED02)==EXPECTED_V02_SHA and sha256(ROOT/PRED04)==EXPECTED_V04_SHA
assert all(x['label_status']=='pre_registered_proposed' for x in holdout)
assert Counter(x['expected_action'] for x in holdout)=={'MOVE_LEFT_TO_RIGHT':24,'HOME':24,'REJECT':24}
def norm(s):return re.sub(r'\s+','',unicodedata.normalize('NFKC',s))
new_set={norm(r['utterance']) for r in holdout}
old_set={norm(r['utterance']) for r in old}
assert len(new_set)==72 and len(old_set)==72 and not new_set.intersection(old_set), '新旧データ重複が検出されました'
assert len({r['case_id'] for r in holdout})==72
print('v06 新規72文：24 MOVE / 24 HOME / 24 REJECT')
print('旧72文との完全一致：0件（空白除去・NFKC正規化）')
print('新規評価文SHA256:',sha256(ROOT/HOLDOUT))
print('正解ラベルの状態：事前提案・人間による最終確認前')
print('一部サンプル：')
for r in (holdout[:2]+holdout[24:26]+holdout[48:50]):
    print(r['case_id'],r['expected_action'],r['utterance'])

## セル2：第7回＋v04の規則を再現し、旧72件でコードの整合性を確認

旧72件は**判定ロジックの再現テスト専用**です。旧データのスコア・marginと候補情報を使い、v02およびv04の記録された72件すべてに一致することを確認します。ここでも新規72文に対してAIは実行しません。

In [ ]:
# v04の拒否ルール（既存の内容と順序を固定）
RULES=[
 ('CANCEL_OR_STOP',r'中止|取り消|取消|キャンセル|やめ(?:て|ろ)|止め(?:て|ろ)|停止|作業禁止'),
 ('NEGATED_MANIPULATION',r'(?:移動|搬送|運搬|運ん|運ば|運ぶ|置い|置く|移載|移し替え|持って|持ち上げ|動か|載せ替え)[^。！？\n]{0,15}(?:しない|しなく|してはいけ|してはだめ|しては駄目|するな|せず|ないで|ほしくありません|禁止)'),
 ('UNSUPPORTED_ACTION',r'切断|溶接|研磨|切削|塗装|加熱|火の中|粉砕|破砕|計測|測定|洗浄|ラベルを貼|投げ|焼き入れ'),
]
COMPILED=[(name,re.compile(pattern)) for name,pattern in RULES]
def safety_gate(text):
    text=unicodedata.normalize('NFKC',str(text))
    for name,pattern in COMPILED:
        m=pattern.search(text)
        if m: return name,m.group(0)
    return 'PASS',''

def lexical_direction_guard(instruction):
    s=re.sub(r'\s+','',instruction)
    if any(w in s for w in ['どちらでも','どちらか','適当に','任意','左か右']): return False
    if any(w in s for w in ['右の円柱','右側の円柱','左の台','左側の台','緑の台へ','緑の台に','右から左']): return False
    src_ok=('左' in s or '緑' in s) and ('円柱' in s or '試料' in s)
    dst_ok=('右' in s or '青' in s) and ('台' in s or '置き場' in s)
    verb_ok=any(w in s for w in ['移動','運ん','搬送','置い','持って'])
    return src_ok and dst_ok and verb_ok

def decide(text, proposed, score, margin, min_score, min_margin, gate=True):
    gate_id, matched=safety_gate(text) if gate else ('PASS','')
    if gate_id!='PASS': return 'REJECT',gate_id,matched
    if score<min_score or margin<min_margin: return 'REJECT',gate_id,matched
    if proposed=='HOME':
        t=re.sub(r'\s+','',text)
        allow=any(w in t for w in ['戻','ホーム','待機']) and not any(w in t for w in ['円柱','運ん','移動','搬送'])
        return ('HOME' if allow else 'REJECT'),gate_id,matched
    if proposed!='MOVE_LEFT_TO_RIGHT':return 'REJECT',gate_id,matched
    return ('MOVE_LEFT_TO_RIGHT' if lexical_direction_guard(text) else 'REJECT'),gate_id,matched

# 実装が第7回とv04の保存結果に一致するか確認。ここでは旧72件以外を使わない。
by04={x['case_id']:x for x in ref04}
m02=[];m04=[]
for r in ref02:
    p=r['proposed_task'];s=float(r['score']);m=float(r['margin']);t=r['utterance']
    if decide(t,p,s,m,.60,.025,False)[0]!=r['prediction']:m02.append(r['case_id'])
    if decide(t,p,s,m,.60,.025,True)[0]!=by04[r['case_id']]['v04_prediction']:m04.append(r['case_id'])
assert not m02 and not m04, f'過去の判定を再現できません：v02={m02} v04={m04}'
print('旧72件の判定再現：v02 72/72、v04 72/72。閾値と拒否ルールは固定')

## セル3：Sentence Transformerを読み込み、新規72件を実際に推論

事前にCSVの新規72件の期待ラベルを確認してください。ラベルに不自然な文があれば**このセルを実行する前に止めて**ください。評価開始後にラベル・閾値を調整すると独立評価ではなくなります。

実行後の結果は当該モデル・当該文面に限定されます。MuJoCoの動作や安全性を実証する実験ではありません。モデル初回取得時は時間がかかります。

In [ ]:
import importlib.util, subprocess
if importlib.util.find_spec('sentence_transformers') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','sentence-transformers'])
from sentence_transformers import SentenceTransformer
import numpy as np
from time import perf_counter
from importlib.metadata import version as pkgversion
MODEL_ID='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
TASK_EXAMPLES={
    'MOVE_LEFT_TO_RIGHT':'左の円柱を右の青い台へ移動して',
    'UNSUPPORTED_REVERSE':'右の円柱を左の緑の台へ移動して',
    'HOME':'ロボットアームを待機姿勢に戻して',
}
TASK_LABELS=list(TASK_EXAMPLES)
print('モデルを読み込みます:',MODEL_ID)
model_text=SentenceTransformer(MODEL_ID,device='cpu')
ref_vectors=model_text.encode([TASK_EXAMPLES[k] for k in TASK_LABELS],normalize_embeddings=True)
start=perf_counter()
vecs=model_text.encode([r['utterance'] for r in holdout],normalize_embeddings=True,show_progress_bar=True)
scores=np.asarray(vecs)@np.asarray(ref_vectors).T
elapsed=perf_counter()-start
raw=[]
for r,sim in zip(holdout,scores):
    order=np.argsort(sim)[::-1]
    ix0,ix1=map(int,order[:2]); raw.append({
        'case_id':r['case_id'], 'utterance':r['utterance'], 'expected_action_proposed':r['expected_action'],
        'proposed_task':TASK_LABELS[ix0], 'score':float(sim[ix0]),
        'margin':float(sim[ix0]-sim[ix1]),
    })
print('新規72件へのモデル推論終了。推論時間（バッチ全体、秒）:',round(elapsed,2))
print('モデルの再学習：なし / MuJoCoの操作：なし')

## セル4：事前固定した2条件で独立評価（基準値と開発データ選定候補）

A：score 0.60 / margin 0.025（v04基準）、B：score 0.35 / margin 0.010（v05で開発用72件から候補選定）。**評価結果に応じた追加の閾値探索は行いません。**

- REJECT誤受理率の分母：事前ラベルREJECTの24件
- 正当指示の過剰拒否率の分母：MOVEとHOMEの48件
- 不適切な搬送許可：本来MOVE以外なのにMOVEと判定した件数

In [ ]:
LABELS=['MOVE_LEFT_TO_RIGHT','HOME','REJECT']
CONFIGS={'A_baseline_v04':(.60,.025),'B_dev_candidate_v05':(.35,.010)}
results=[]
for rawrow in raw:
    r=dict(rawrow)
    for name,(score_min,margin_min) in CONFIGS.items():
        p,gate,matched=decide(r['utterance'],r['proposed_task'],r['score'],r['margin'],score_min,margin_min,True)
        r[name+'_prediction']=p
        r[name+'_gate_rule']=gate
        r[name+'_gate_match']=matched
    results.append(r)

def wilson_95(k,n):
    if n==0:return [None,None]
    z=1.959963984540054
    p=k/n; denom=1+z*z/n
    mid=(p+z*z/(2*n))/denom
    half=z*math.sqrt((p*(1-p)/n)+(z*z/(4*n*n)))/denom
    return [max(0,mid-half),min(1,mid+half)]

def evaluate(rows,config):
    col=config+'_prediction'
    truth=[r['expected_action_proposed'] for r in rows]
    pred=[r[col] for r in rows]
    cm=[[sum(x==a and y==b for x,y in zip(truth,pred)) for b in LABELS] for a in LABELS]
    perclass={}
    for i,label in enumerate(LABELS):
        tp=cm[i][i]; actual=sum(cm[i]); predicted=sum(row[i] for row in cm)
        pre=tp/predicted if predicted else 0.0
        rec=tp/actual if actual else 0.0
        f1=2*pre*rec/(pre+rec) if pre+rec else 0.0
        perclass[label]={'precision':pre,'recall':rec,'f1':f1,'support':actual}
    false_accept=sum(a=='REJECT' and b!='REJECT' for a,b in zip(truth,pred))
    false_reject=sum(a!='REJECT' and b=='REJECT' for a,b in zip(truth,pred))
    unintended_move=sum(a!='MOVE_LEFT_TO_RIGHT' and b=='MOVE_LEFT_TO_RIGHT' for a,b in zip(truth,pred))
    correct=sum(a==b for a,b in zip(truth,pred))
    return {
        'n':len(rows),'correct':correct,'accuracy':correct/len(rows),
        'accuracy_wilson95':wilson_95(correct,len(rows)),
        'macro_f1':sum(x['f1'] for x in perclass.values())/3,
        'class_metrics':perclass,'labels':LABELS,'confusion_matrix':cm,
        'reject_false_accept':false_accept,'reject_false_accept_denominator':24,
        'reject_false_accept_wilson95':wilson_95(false_accept,24),
        'valid_false_reject':false_reject,'valid_false_reject_denominator':48,
        'unintended_move_count':unintended_move,
    }
metrics={name:evaluate(results,name) for name in CONFIGS}
for name,m in metrics.items():
    print('\n',name,'固定設定=',CONFIGS[name])
    print(f"正答 {m['correct']}/72 = {m['accuracy']:.1%}、Macro-F1={m['macro_f1']:.3f}")
    print('REJECT誤受理',m['reject_false_accept'],'/24、正当指示の過剰拒否',m['valid_false_reject'],'/48')
    print('不適切なMOVE許可',m['unintended_move_count'],'件')
    print('混同行列：行=期待 / 列=予測 [MOVE, HOME, REJECT]')
    for row in m['confusion_matrix']:print(row)
print('\nラベルは暫定。これは合成新規72件に対する分類精度であり、実機や実世界の安全性ではありません。')

## セル5：誤判定を一覧にし、実験条件・SHA256を記録

評価後に文面・正解ラベル・閾値を変更せず、誤判定はすべてCSVに残します。改良方針の検討は**別バージョンv07以降**で行います。

In [ ]:
from datetime import datetime,timezone
OUT_PRED='08_v06_independent_predictions.csv'
OUT_ERRORS='08_v06_misclassifications.csv'
OUT_SUMMARY='08_v06_evaluation_summary.json'
OUT_ZIP='PhysicalAI_08_v06_results.zip'
now=datetime.now(timezone.utc).isoformat()
fieldnames=list(results[0])
with (ROOT/OUT_PRED).open('w',newline='',encoding='utf-8-sig') as f:
    w=csv.DictWriter(f,fieldnames=fieldnames);w.writeheader()
    for item in results:
        # 表示桁数は小数6桁、判定自体は浮動小数の原値を使用
        line=dict(item);line['score']=f"{item['score']:.6f}";line['margin']=f"{item['margin']:.6f}"
        w.writerow(line)
errors=[r for r in results if any(r[name+'_prediction']!=r['expected_action_proposed'] for name in CONFIGS)]
with (ROOT/OUT_ERRORS).open('w',newline='',encoding='utf-8-sig') as f:
    w=csv.DictWriter(f,fieldnames=fieldnames);w.writeheader();w.writerows(errors)
manifest={
    'version':VERSION,'date_utc':now,
    'evaluation_type':'holdout_synthetic_unseen_utterances',
    'label_status':'proposed_not_independently_adjudicated',
    'dataset_test_count':len(holdout),
    'dataset_sha256':sha256(ROOT/HOLDOUT),
    'old_v02_pred_sha256':sha256(ROOT/PRED02),
    'old_v04_replay_sha256':sha256(ROOT/PRED04),
    'model_id':MODEL_ID,
    'task_examples':TASK_EXAMPLES,
    'thresholds_precommitted':{k: {'score':v[0], 'margin':v[1]} for k,v in CONFIGS.items()},
    'library_versions':{'python':platform.python_version(), 'numpy':np.__version__,
        'sentence_transformers':pkgversion('sentence-transformers')},
    'results':metrics,
    'important_limitations':['人間の最終ラベル確認前','人工作成テキストで実環境代表性なし',
        '閾値候補Bは旧72件で選定済み','運動・接触シミュレーションの評価ではない']
}
(ROOT/OUT_SUMMARY).write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('保存:',OUT_PRED,len(results),'行')
print('両条件の少なくとも一方で誤判定:',OUT_ERRORS,len(errors),'行')
print('保存:',OUT_SUMMARY)

## セル6：結果3ファイルと実験入力CSVをZIPにまとめてWindowsへ保存

このセルを実行すると、v06の解析結果ZIPがダウンロードされます。**Notebookは別途、Colab「ファイル」→「.ipynb をダウンロード」**で保存してください。前版のファイルと混同しないようv06という名前を維持します。GitHubへの自動登録はしません。

In [ ]:
with zipfile.ZipFile(ROOT/OUT_ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for fn in [HOLDOUT,OLD,'08_v06_old72_label_audit.csv',OUT_PRED,OUT_ERRORS,OUT_SUMMARY]:
        if (ROOT/fn).exists():z.write(ROOT/fn,arcname=fn)
print('出力ZIP:',ROOT/OUT_ZIP)
try:
    from google.colab import files
    files.download(str(ROOT/OUT_ZIP))
except ImportError:
    print('Colab外ではZIPを手動で取得してください。')

## v06の結論を記述する際の原則

- v05の **55.6%→61.1%→72.2%** は旧72件に対する開発データ内の数値です。v06結果とは混ぜない。
- v06の新規72件の評価は「独立した新規合成文」ではあるが、事前ラベルが人間により確定するまでは**暫定結果**として報告。
- 不適切指示を1件も受理しなかったとしても、その集合内の結果にすぎず、実機の安全を保証しない。
- 出力結果が期待に反しても、v06で評価文・閾値・コードを編集しない。改修するならv07として別ノートブックを作る。
- 第8回の質量・摩擦係数を変えたMuJoCo試験45条件は**未実施**であり、このNotebookには含まない。